# LeetCode 468 — Validate IP Address bằng Python

Link đề: https://leetcode.com/problems/validate-ip-address/ · Độ khó: **Medium** · Chủ đề: String, Simulation, Parsing, Regex

Skeleton LeetCode:

```python
class Solution:
    def validIPAddress(self, queryIP: str) -> str:
        ...
```

Ghi chú: đề được viết lại bằng tiếng Việt theo đề gốc; mọi đáp án đều được đối chiếu với một "oracle" độc lập dựng từ thư viện `ipaddress` của Python và với bộ sinh địa chỉ hợp lệ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: phân loại một chuỗi là địa chỉ **IPv4**, **IPv6** hay **Neither** (không phải cả hai) theo đúng các luật của đề (không số 0 đứng đầu ở IPv4, nhóm hex dài 1–4 ở IPv6, cho phép số 0 đứng đầu ở IPv6).
2. Dùng thạo các công cụ chuỗi của Python: `split`, `in`, `all`, `isdigit`, `int(...)` (và cái bẫy của nó), cắt lát, và thư viện `re` (biểu thức chính quy) cơ bản.
3. Biết **tách bài toán thành hai bộ kiểm tra độc lập** (`is_ipv4`, `is_ipv6`) rồi chọn nhánh dựa vào dấu phân tách `.` hay `:`.
4. Cài đặt theo 5 cách khác nhau về ý tưởng: **dùng `int()` (có lỗi) → kiểm tra từng ký tự sau `split` → regex → máy trạng thái quét một lượt không `split` → dùng thư viện `ipaddress` có lọc trước**, và so sánh ưu nhược điểm.
5. Tránh các lỗi kinh điển: `int("0x1f", 16)` vẫn chạy, `isdigit()` nhận cả chữ số Unicode, quên số 0 đứng đầu, quên nhóm rỗng khi `split`, giới hạn 255/4 ký tự.

## 2. Đề bài & ví dụ

Cho chuỗi `queryIP`. Trả về:

- `"IPv4"` nếu `queryIP` là địa chỉ IPv4 hợp lệ;
- `"IPv6"` nếu là địa chỉ IPv6 hợp lệ;
- `"Neither"` nếu không phải địa chỉ hợp lệ nào.

**IPv4 hợp lệ** có dạng `x1.x2.x3.x4` với `0 <= xi <= 255` và `xi` **không có số 0 đứng đầu**. Ví dụ `"192.168.1.1"`, `"192.168.1.0"` hợp lệ; `"192.168.01.1"`, `"192.168.1.00"`, `"192.168@1.1"` không hợp lệ.

**IPv6 hợp lệ** có dạng `x1:x2:x3:x4:x5:x6:x7:x8` trong đó:

- `1 <= xi.length <= 4`;
- `xi` là chuỗi **thập lục phân (hex)**: chữ số, chữ `a`–`f`, chữ `A`–`F`;
- **cho phép** số 0 đứng đầu.

Ví dụ hợp lệ: `"2001:0db8:85a3:0000:0000:8a2e:0370:7334"`, `"2001:db8:85a3:0:0:8A2E:0370:7334"`. Không hợp lệ: `"2001:0db8:85a3::8A2E:037j:7334"`, `"02001:0db8:85a3:0000:0000:8a2e:0370:7334"`.

**Ví dụ 1:** `"172.16.254.1"` → `"IPv4"`.

**Ví dụ 2:** `"2001:0db8:85a3:0:0:8A2E:0370:7334"` → `"IPv6"`.

**Ví dụ 3:** `"256.256.256.256"` → `"Neither"`.

**Ràng buộc:** `queryIP` chỉ gồm chữ cái tiếng Anh, chữ số và hai ký tự `'.'`, `':'`.

In [1]:
import ipaddress
import random
import re
import time

examples = [
    ("172.16.254.1", "IPv4"),
    ("2001:0db8:85a3:0:0:8A2E:0370:7334", "IPv6"),
    ("256.256.256.256", "Neither"),
]
for q, expected in examples:
    print(f"{q!r:40} -> {expected}")

'172.16.254.1'                           -> IPv4
'2001:0db8:85a3:0:0:8A2E:0370:7334'      -> IPv6
'256.256.256.256'                        -> Neither


## 3. Kiến thức nền

### 3.1. `str.split(sep)` — cắt chuỗi thành danh sách

`"a.b.c".split(".")` cho `["a", "b", "c"]`. **Chú ý:** nếu có hai dấu phân tách liền nhau hoặc dấu ở đầu/cuối, ta nhận về **chuỗi rỗng `""`** trong danh sách. Đây chính là cách ta phát hiện "nhóm rỗng".

In [2]:
print("1.2.3.4".split("."))       # ['1', '2', '3', '4']
print("1..3.4".split("."))        # ['1', '', '3', '4']   <- có nhóm rỗng
print("1.2.3.4.".split("."))      # ['1', '2', '3', '4', '']   <- dấu chấm thừa ở cuối -> 5 nhóm
print("".split("."))              # ['']  -> chuỗi rỗng cho đúng 1 nhóm rỗng
print(len("1.2.3".split(".")))    # 3

['1', '2', '3', '4']
['1', '', '3', '4']
['1', '2', '3', '4', '']
['']
3


### 3.2. Toán tử `in` với chuỗi, và `all(...)` với generator

`ch in "0123456789"` kiểm tra ký tự `ch` có nằm trong chuỗi mẫu không. `all(điều_kiện for ... in ...)` trả `True` nếu **mọi** phần tử thỏa điều kiện (và `True` với dãy rỗng!). Ta dùng chúng để kiểm tra "mọi ký tự đều là chữ số".

In [3]:
CHU_SO = "0123456789"
print("7" in CHU_SO, "x" in CHU_SO)                          # True False
print(all(ch in CHU_SO for ch in "2024"))                    # True
print(all(ch in CHU_SO for ch in "20x4"))                    # False
print(all(ch in CHU_SO for ch in ""))                        # True  <- dãy rỗng: nhớ kiểm tra độ dài riêng

True False
True
False
True


### 3.3. `int(chuỗi)` và `int(chuỗi, 16)` — và vì sao nên cẩn thận

`int("255")` ra `255`; `int("ff", 16)` ra `255` (đọc theo hệ 16). Nhưng `int` **dễ dãi hơn ta tưởng**: nó chấp nhận dấu `+`/`-`, khoảng trắng hai đầu, dấu gạch dưới `_`, và với cơ số 16 còn chấp nhận tiền tố `0x`. Vì vậy chỉ gọi `int` **sau khi** đã kiểm tra từng ký tự.

In [4]:
print(int("255"), int("ff", 16), int("FF", 16))     # 255 255 255
print(int(" 42 "), int("+7"), int("1_0"))            # 42 7 10   <- dễ dãi
print(int("0x1f", 16))                               # 31        <- 'x' cũng là chữ cái hợp lệ trong đề, nhưng KHÔNG phải hex!
print(int("007"))                                    # 7         <- số 0 đứng đầu bị "nuốt"

255 255 255
42 7 10
31
7


### 3.4. `isdigit()` — cũng đừng tin hoàn toàn

`"123".isdigit()` là `True`, nhưng `isdigit` còn nhận cả một số ký tự Unicode như `"²"` (số mũ), `"٣"` (chữ số Ả Rập). Đề bài chỉ có chữ số thường nên ít gặp, nhưng kiểm tra `ch in "0123456789"` an toàn hơn.

In [5]:
print("123".isdigit(), "²".isdigit(), "٣".isdigit())      # True True True
print("²" in "0123456789")                                 # False
try:
    int("²")
except ValueError as e:
    print("int('²') lỗi:", e)

True True True
False
int('²') lỗi: invalid literal for int() with base 10: '²'


### 3.5. Biểu thức chính quy (regex) tối thiểu

Thư viện `re` mô tả mẫu chuỗi bằng ký hiệu. Ta chỉ cần vài ký hiệu:

- `[0-9a-fA-F]` — một ký tự trong tập; `{1,4}` — lặp 1 đến 4 lần; `{3}` — đúng 3 lần.
- `(A|B)` — chọn A hoặc B; `(...)` — nhóm; `\.` — dấu chấm thật (`.` trần nghĩa là "ký tự bất kỳ").
- `re.fullmatch(mẫu, chuỗi)` — **cả chuỗi** phải khớp mẫu (an toàn hơn `match`/`search`).

In [6]:
print(bool(re.fullmatch(r"[0-9a-fA-F]{1,4}", "0db8")))       # True
print(bool(re.fullmatch(r"[0-9a-fA-F]{1,4}", "0db8f")))      # False  (5 ký tự)
print(bool(re.fullmatch(r"([0-9]{1,3}\.){3}[0-9]{1,3}", "1.22.333.4")))   # True
print(bool(re.fullmatch(r"[0-9]+", "12\n")))                 # False: fullmatch không bỏ qua '\n' thừa như `$`

True
False
True
False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

**Bước 1 — nhìn dấu phân tách để biết định hướng.** Có `.` → chỉ có thể là IPv4. Có `:` → chỉ có thể là IPv6. Có cả hai, hay không có cái nào → `Neither`. (Nhưng ta không cần phân nhánh kỹ: chỉ cần chạy hai bộ kiểm tra; cái nào đúng thì trả về.)

**Bước 2 — IPv4: bốn nhóm.** Tính tay `"192.168.01.1"`: tách bằng `.` ra `['192', '168', '01', '1']`: đúng 4 nhóm. Xét từng nhóm: `192` ✔ (không 0 đầu, ≤ 255), `168` ✔, `01` ✘ (dài 2 mà bắt đầu bằng `0`). → `Neither`.

**Bước 3 — IPv6: tám nhóm.** Tính tay `"2001:db8:85a3:0:0:8A2E:0370:7334"`: tách bằng `:` ra 8 nhóm `['2001','db8','85a3','0','0','8A2E','0370','7334']`; mỗi nhóm dài 1–4, mọi ký tự đều hex ✔ → `IPv6`. Còn `"2001:0db8:85a3::8A2E:037j:7334"`: tách ra 7 nhóm (một nhóm rỗng), và `037j` có `j` không phải hex → `Neither`.

**Luật tóm gọn:**

| | IPv4 | IPv6 |
|---|---|---|
| Phân tách | `.` | `:` |
| Số nhóm | đúng 4 | đúng 8 |
| Mỗi nhóm | 1–3 chữ số thập phân, giá trị 0–255, không 0 đầu (trừ chính `"0"`) | 1–4 ký tự hex, **cho phép** 0 đầu |

In [7]:
# Tính tay bằng code nhỏ
for q in ["192.168.01.1", "2001:db8:85a3:0:0:8A2E:0370:7334", "2001:0db8:85a3::8A2E:037j:7334"]:
    sep = "." if "." in q else ":"
    nhom = q.split(sep)
    print(len(nhom), nhom)

4 ['192', '168', '01', '1']
8 ['2001', 'db8', '85a3', '0', '0', '8A2E', '0370', '7334']
7 ['2001', '0db8', '85a3', '', '8A2E', '037j', '7334']


## 5. Cách 1 — Brute Force / baseline: dựa vào `int()` (cách "nhanh tay", có lỗi)

**Ý tưởng:** cách nghĩ tự nhiên là để Python làm hộ việc đọc số:

- IPv4: tách 4 nhóm; với mỗi nhóm đổi bằng `int`; hợp lệ nếu `0 <= số <= 255` và **dạng viết chuẩn** `str(số) == nhóm` (cách này loại luôn số 0 đầu, dấu `+`, khoảng trắng...).
- IPv6: tách 8 nhóm; mỗi nhóm có độ dài 1–4 và `int(nhóm, 16)` **không lỗi**.

Nghe hợp lý, nhưng phần IPv6 ẩn chứa một cái bẫy (mục 3.3): `int("0x1f", 16)` **không lỗi**.

In [8]:
def naive_ipv4(s):
    parts = s.split(".")
    if len(parts) != 4:
        return False
    for t in parts:
        try:
            v = int(t)
        except ValueError:
            return False
        if not 0 <= v <= 255 or str(v) != t:      # dạng chuẩn: loại 0 đầu, '+', ' ' ...
            return False
    return True

def naive_ipv6(s):
    parts = s.split(":")
    if len(parts) != 8:
        return False
    for t in parts:
        if not 1 <= len(t) <= 4:
            return False
        try:
            int(t, 16)                             # BẪY: "0x1f", "+1f", " 1f" vẫn qua
        except ValueError:
            return False
    return True

def naive(q):
    if naive_ipv4(q):
        return "IPv4"
    if naive_ipv6(q):
        return "IPv6"
    return "Neither"

for q, expected in examples:
    print(q, naive(q), expected)
    assert naive(q) == expected

172.16.254.1 IPv4 IPv4
2001:0db8:85a3:0:0:8A2E:0370:7334 IPv6 IPv6
256.256.256.256 Neither Neither


In [9]:
# Nút thắt: lỗi đúng với đề (chữ 'x' nằm trong "English letters" nhưng không phải chữ số hex)
bad = "0x1f:0db8:85a3:0:0:8A2E:0370:7334"
print(naive(bad), "<- SAI, đúng phải là Neither")
assert naive(bad) == "IPv6"

IPv6 <- SAI, đúng phải là Neither


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (danh sách nhóm sau `split`).

**Nút thắt:** hàm `int` quá dễ dãi nên **không thể dùng nó làm bộ kiểm tra cú pháp**. Cần kiểm tra **từng ký tự** có nằm trong tập cho phép hay không. Đó là cách 2.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Tách việc:** viết riêng `is_ipv4` và `is_ipv6`, mỗi hàm trả `True/False`; hàm chính chỉ gọi lần lượt. Hai địa chỉ dùng dấu phân tách khác nhau nên không thể cùng hợp lệ.
2. **Mỗi nhóm kiểm tra bằng luật cục bộ, không "thử đọc":** độ dài trong khoảng cho phép; mọi ký tự thuộc tập cho phép (`0123456789` hoặc `0123456789abcdefABCDEF`); với IPv4 thêm: không số 0 đầu và giá trị ≤ 255. Chỉ gọi `int` **sau** khi chắc chắn toàn chữ số thập phân.
3. **Nhóm rỗng tự bị loại** bởi điều kiện độ dài `>= 1` (kể cả trường hợp dấu phân tách thừa ở đầu/giữa/cuối).
4. Mọi cách đều `O(n)`; khác biệt chủ yếu ở **cách biểu diễn luật**: code thủ công, regex, hay máy trạng thái.

## 7. Các cách tiếp theo

### Cách 2 — `split` + kiểm tra từng ký tự (cách chuẩn)

**Giải thích từng đoạn:**

- `is_ipv4(s)`: `parts = s.split(".")`; không đủ 4 nhóm → `False`. Với mỗi nhóm `t`: độ dài trong `[1, 3]`; mọi ký tự trong `CHU_SO`; nếu `len(t) > 1` thì `t[0] != "0"`; `int(t) <= 255`.
- `is_ipv6(s)`: `parts = s.split(":")`; không đủ 8 nhóm → `False`. Mỗi nhóm: độ dài `[1, 4]` và mọi ký tự trong `HEX`.
- Hàm chính: `is_ipv4` → `"IPv4"`; `is_ipv6` → `"IPv6"`; còn lại `"Neither"`.

In [10]:
CHU_SO = "0123456789"
HEX = "0123456789abcdefABCDEF"

def is_ipv4(s):
    parts = s.split(".")
    if len(parts) != 4:
        return False
    for t in parts:
        if not 1 <= len(t) <= 3:                       # loại nhóm rỗng và nhóm quá dài
            return False
        if any(ch not in CHU_SO for ch in t):          # chỉ chữ số thập phân
            return False
        if len(t) > 1 and t[0] == "0":                 # không số 0 đứng đầu (trừ chính "0")
            return False
        if int(t) > 255:
            return False
    return True

def is_ipv6(s):
    parts = s.split(":")
    if len(parts) != 8:
        return False
    for t in parts:
        if not 1 <= len(t) <= 4:
            return False
        if any(ch not in HEX for ch in t):
            return False
    return True

def split_check(q):
    if is_ipv4(q):
        return "IPv4"
    if is_ipv6(q):
        return "IPv6"
    return "Neither"

for q, expected in examples:
    print(q, split_check(q), expected)
    assert split_check(q) == expected
print(split_check("0x1f:0db8:85a3:0:0:8A2E:0370:7334"))      # Neither (đã sửa lỗi của Cách 1)

172.16.254.1 IPv4 IPv4
2001:0db8:85a3:0:0:8A2E:0370:7334 IPv6 IPv6
256.256.256.256 Neither Neither
Neither


In [11]:
class SolutionSplit:
    def validIPAddress(self, queryIP: str) -> str:
        digits = "0123456789"
        hexs = "0123456789abcdefABCDEF"

        def is_ipv4(s: str) -> bool:
            parts = s.split(".")
            if len(parts) != 4:
                return False
            for t in parts:
                if not 1 <= len(t) <= 3 or any(c not in digits for c in t):
                    return False
                if len(t) > 1 and t[0] == "0":
                    return False
                if int(t) > 255:
                    return False
            return True

        def is_ipv6(s: str) -> bool:
            parts = s.split(":")
            if len(parts) != 8:
                return False
            return all(1 <= len(t) <= 4 and all(c in hexs for c in t) for t in parts)

        if is_ipv4(queryIP):
            return "IPv4"
        if is_ipv6(queryIP):
            return "IPv6"
        return "Neither"

print(SolutionSplit().validIPAddress("172.16.254.1"))

IPv4


**Độ phức tạp (Cách 2):** `O(n)` thời gian (mỗi ký tự được xét một số lần hằng số), `O(n)` bộ nhớ cho danh sách nhóm.

### Cách 3 — Biểu thức chính quy (regex)

**Ý tưởng khác:** mô tả **hình dạng** hợp lệ bằng một mẫu và để thư viện `re` kiểm tra.

- Một nhóm IPv4 (0–255, không 0 đầu): `25[0-5]` hoặc `2[0-4][0-9]` hoặc `1[0-9][0-9]` hoặc `[1-9]?[0-9]` (một chữ số, hoặc chữ số 1–9 rồi một chữ số: nên không có `01`).
- IPv4 = `nhóm\.nhóm\.nhóm\.nhóm`.
- Một nhóm IPv6: `[0-9a-fA-F]{1,4}`; IPv6 = bảy nhóm có `:` theo sau rồi một nhóm cuối.

Dùng `fullmatch` (cả chuỗi phải khớp) và viết `[0-9]` thay vì `\d` (vì `\d` trong Python nhận cả chữ số Unicode).

In [12]:
nhom4 = r"(25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])"
RE_V4 = re.compile(nhom4 + r"(\." + nhom4 + r"){3}")
RE_V6 = re.compile(r"([0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}")

def regex_check(q):
    if RE_V4.fullmatch(q):
        return "IPv4"
    if RE_V6.fullmatch(q):
        return "IPv6"
    return "Neither"

for q, expected in examples:
    print(q, regex_check(q), expected)
    assert regex_check(q) == expected
print(regex_check("192.168.01.1"), regex_check("192.168.1.00"), regex_check("1.1.1.1."))

172.16.254.1 IPv4 IPv4
2001:0db8:85a3:0:0:8A2E:0370:7334 IPv6 IPv6
256.256.256.256 Neither Neither
Neither Neither Neither


In [13]:
import re as _re

class SolutionRegex:
    _OCT = r"(25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])"
    _V4 = _re.compile(_OCT + r"(\." + _OCT + r"){3}")
    _V6 = _re.compile(r"([0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}")

    def validIPAddress(self, queryIP: str) -> str:
        if self._V4.fullmatch(queryIP):
            return "IPv4"
        if self._V6.fullmatch(queryIP):
            return "IPv6"
        return "Neither"

print(SolutionRegex().validIPAddress("2001:0db8:85a3:0:0:8A2E:0370:7334"))

IPv6


**Độ phức tạp (Cách 3):** mẫu không có lặp lồng nhau nguy hiểm nên khớp trong `O(n)` thời gian, `O(1)` bộ nhớ phụ (không tạo danh sách). Ưu điểm: ngắn. Nhược điểm: khó đọc, khó gỡ lỗi, dễ sai vài ký tự trong mẫu (ví dụ quên `{1,4}` hay `fullmatch`).

### Cách 4 — Máy trạng thái / quét một lượt, **không** `split`

**Ý tưởng khác:** đi dọc chuỗi bằng một chỉ số `i`; mỗi vòng ta "ăn" một nhóm: đọc liên tiếp các ký tự hợp lệ, đếm độ dài (và tích lũy giá trị với IPv4), rồi **nhìn ký tự tiếp theo**: phải là dấu phân tách (và chưa đủ số nhóm) hoặc hết chuỗi (và đã đủ nhóm). Không tạo danh sách trung gian nên chỉ tốn `O(1)` bộ nhớ.

**Giải thích từng đoạn (IPv4):** `groups` đếm số nhóm đã đọc; vòng `while True`: `j = i`, `val = 0`; chừng nào `s[j]` là chữ số thì `val = val*10 + chữ_số`, `j += 1`; `length = j - i`. Kiểm tra: `1 <= length <= 3`, `(length == 1 or s[i] != "0")`, `val <= 255`. Tăng `groups`; nếu `j == n`: hợp lệ khi `groups == 4`; nếu `s[j] != "."` hoặc `groups == 4` thì sai; ngược lại `i = j + 1` đọc nhóm kế.

In [14]:
def scan_ipv4(s):
    n = len(s)
    i = 0
    groups = 0
    while True:
        j = i
        val = 0
        while j < n and s[j] in CHU_SO:
            val = val * 10 + (ord(s[j]) - 48)
            j += 1
        length = j - i
        if not 1 <= length <= 3 or (length > 1 and s[i] == "0") or val > 255:
            return False
        groups += 1
        if j == n:
            return groups == 4
        if s[j] != "." or groups == 4:
            return False
        i = j + 1

def scan_ipv6(s):
    n = len(s)
    i = 0
    groups = 0
    while True:
        j = i
        while j < n and s[j] in HEX:
            j += 1
        if not 1 <= j - i <= 4:
            return False
        groups += 1
        if j == n:
            return groups == 8
        if s[j] != ":" or groups == 8:
            return False
        i = j + 1

def scan_check(q):
    if scan_ipv4(q):
        return "IPv4"
    if scan_ipv6(q):
        return "IPv6"
    return "Neither"

for q, expected in examples:
    print(q, scan_check(q), expected)
    assert scan_check(q) == expected
print(scan_check(""), scan_check("1.1.1.1."), scan_check("1.1.1.1.1"))

172.16.254.1 IPv4 IPv4
2001:0db8:85a3:0:0:8A2E:0370:7334 IPv6 IPv6
256.256.256.256 Neither Neither
Neither Neither Neither


In [15]:
class SolutionScan:
    def validIPAddress(self, queryIP: str) -> str:
        s, n = queryIP, len(queryIP)
        digits, hexs = "0123456789", "0123456789abcdefABCDEF"

        def scan(sep: str, alphabet: str, groups_needed: int, max_len: int, decimal: bool) -> bool:
            i = groups = 0
            while True:
                j, val = i, 0
                while j < n and s[j] in alphabet:
                    val = val * 10 + ord(s[j]) - 48
                    j += 1
                length = j - i
                if not 1 <= length <= max_len:
                    return False
                if decimal and ((length > 1 and s[i] == "0") or val > 255):
                    return False
                groups += 1
                if j == n:
                    return groups == groups_needed
                if s[j] != sep or groups == groups_needed:
                    return False
                i = j + 1

        if scan(".", digits, 4, 3, True):
            return "IPv4"
        if scan(":", hexs, 8, 4, False):
            return "IPv6"
        return "Neither"

print(SolutionScan().validIPAddress("256.256.256.256"), SolutionScan().validIPAddress("1.2.3.4"))

Neither IPv4


**Độ phức tạp (Cách 4):** mỗi ký tự được đọc đúng một lần → `O(n)` thời gian, `O(1)` bộ nhớ phụ. Đánh đổi: dài hơn và dễ sai chỉ số hơn `split`.

### Cách 5 — Dùng thư viện `ipaddress` có lọc trước

**Ý tưởng khác:** Python đã có sẵn `ipaddress.IPv4Address` và `IPv6Address` ném lỗi nếu chuỗi sai. Nhưng thư viện này **dễ dãi hơn đề**: IPv6 nó cho phép viết tắt `::` và hậu tố `%eth0`, IPv6 nhúng IPv4... nên ta phải **lọc trước** theo đúng "hình dạng" của đề: IPv4 phải có đúng 3 dấu `.` và chỉ chữ số/dấu chấm; IPv6 phải có đúng 7 dấu `:`, không có `::`, chỉ ký tự hex/dấu hai chấm. Sau lọc, giao cho thư viện kiểm tra phần còn lại.

> Lưu ý: kết quả của `IPv4Address` với số 0 đứng đầu phụ thuộc phiên bản Python (từ 3.9.5 từ chối). Trên LeetCode phiên bản có thể khác, nên **cách này chỉ để tham khảo/đối chiếu**, không nên nộp.

In [16]:
def lib_check(q):
    if q.count(".") == 3 and all(c in "0123456789." for c in q):
        try:
            ipaddress.IPv4Address(q)
            return "IPv4"
        except ValueError:
            return "Neither"
    if q.count(":") == 7 and "::" not in q and all(c in HEX + ":" for c in q):
        try:
            ipaddress.IPv6Address(q)
            return "IPv6"
        except ValueError:
            return "Neither"
    return "Neither"

for q, expected in examples:
    print(q, lib_check(q), expected)
    assert lib_check(q) == expected
print(lib_check("192.168.01.1"), lib_check("0x1f:0db8:85a3:0:0:8A2E:0370:7334"))

172.16.254.1 IPv4 IPv4
2001:0db8:85a3:0:0:8A2E:0370:7334 IPv6 IPv6
256.256.256.256 Neither Neither
Neither Neither


In [17]:
import ipaddress as _ip

class SolutionLib:
    def validIPAddress(self, queryIP: str) -> str:
        q = queryIP
        try:
            if q.count(".") == 3 and all(c in "0123456789." for c in q):
                _ip.IPv4Address(q)
                return "IPv4"
            if q.count(":") == 7 and "::" not in q and all(c in "0123456789abcdefABCDEF:" for c in q):
                _ip.IPv6Address(q)
                return "IPv6"
        except ValueError:
            pass
        return "Neither"

print(SolutionLib().validIPAddress("172.16.254.1"), SolutionLib().validIPAddress("1.1.1"))

IPv4 Neither


**Độ phức tạp (Cách 5):** `O(n)` thời gian, `O(n)` bộ nhớ. Ưu điểm: tận dụng thư viện chuẩn, hữu ích khi viết ứng dụng thật. Nhược điểm: không hoàn toàn trùng luật của đề nếu quên lọc; phụ thuộc phiên bản; bài phỏng vấn thường cấm.

## 8. TRACE từng bước

Trace **Cách 2** (`is_ipv4` và `is_ipv6`) từng nhóm.

**Ví dụ A — `"192.168.01.1"`:**

| nhóm | độ dài | toàn chữ số? | 0 đầu? | `int` | kết luận |
|---|---|---|---|---|---|
| `192` | 3 | ✔ | không | 192 ≤ 255 | tiếp tục |
| `168` | 3 | ✔ | không | 168 ≤ 255 | tiếp tục |
| `01` | 2 | ✔ | **có** (`t[0]=='0'`, dài 2) | — | **`False`** → `Neither` |

**Ví dụ B — `"2001:0db8:85a3:0:0:8A2E:0370:7334"`:** 8 nhóm; mọi nhóm dài 1–4 và toàn ký tự hex → `IPv6`.

**Ví dụ C — `"2001:0db8:85a3::8A2E:037j:7334"`:** `split(":")` ra 7 nhóm (do `::` sinh nhóm rỗng, số nhóm = 7 chứ không phải 8) → `Neither` ngay từ bước đếm.

In [18]:
def trace_ipv4(s):
    parts = s.split(".")
    print(f"split('.') -> {parts}  (số nhóm = {len(parts)})")
    if len(parts) != 4:
        print("=> không đủ 4 nhóm: False")
        return False
    for t in parts:
        ok_len = 1 <= len(t) <= 3
        ok_digit = ok_len and all(c in CHU_SO for c in t)
        zero_dau = len(t) > 1 and t[0] == "0"
        gia_tri = int(t) if ok_digit else None
        print(f"  nhóm {t!r:>6}: dài_ok={ok_len}, toàn_số={ok_digit}, 0_đầu={zero_dau}, giá_trị={gia_tri}")
        if not ok_len or not ok_digit or zero_dau or gia_tri > 255:
            print("  => nhóm này sai: False")
            return False
    print("=> mọi nhóm đều ổn: True")
    return True

def trace_ipv6(s):
    parts = s.split(":")
    print(f"split(':') -> {parts}  (số nhóm = {len(parts)})")
    if len(parts) != 8:
        print("=> không đủ 8 nhóm: False")
        return False
    for t in parts:
        ok_len = 1 <= len(t) <= 4
        ok_hex = ok_len and all(c in HEX for c in t)
        print(f"  nhóm {t!r:>7}: dài_ok={ok_len}, toàn_hex={ok_hex}")
        if not ok_hex:
            print("  => nhóm này sai: False")
            return False
    print("=> mọi nhóm đều ổn: True")
    return True

trace_ipv4("192.168.01.1")
print()
trace_ipv6("2001:0db8:85a3:0:0:8A2E:0370:7334")
print()
trace_ipv6("2001:0db8:85a3::8A2E:037j:7334")

split('.') -> ['192', '168', '01', '1']  (số nhóm = 4)
  nhóm  '192': dài_ok=True, toàn_số=True, 0_đầu=False, giá_trị=192
  nhóm  '168': dài_ok=True, toàn_số=True, 0_đầu=False, giá_trị=168
  nhóm   '01': dài_ok=True, toàn_số=True, 0_đầu=True, giá_trị=1
  => nhóm này sai: False

split(':') -> ['2001', '0db8', '85a3', '0', '0', '8A2E', '0370', '7334']  (số nhóm = 8)
  nhóm  '2001': dài_ok=True, toàn_hex=True
  nhóm  '0db8': dài_ok=True, toàn_hex=True
  nhóm  '85a3': dài_ok=True, toàn_hex=True
  nhóm     '0': dài_ok=True, toàn_hex=True
  nhóm     '0': dài_ok=True, toàn_hex=True
  nhóm  '8A2E': dài_ok=True, toàn_hex=True
  nhóm  '0370': dài_ok=True, toàn_hex=True
  nhóm  '7334': dài_ok=True, toàn_hex=True
=> mọi nhóm đều ổn: True

split(':') -> ['2001', '0db8', '85a3', '', '8A2E', '037j', '7334']  (số nhóm = 7)
=> không đủ 8 nhóm: False


False

In [19]:
# Trace thêm Cách 4 (máy trạng thái): in từng nhóm được "ăn"
def trace_scan_ipv4(s):
    n, i, groups = len(s), 0, 0
    while True:
        j, val = i, 0
        while j < n and s[j] in CHU_SO:
            val = val * 10 + (ord(s[j]) - 48)
            j += 1
        length = j - i
        print(f"  nhóm #{groups + 1}: s[{i}:{j}] = {s[i:j]!r}, dài {length}, giá trị {val}")
        if not 1 <= length <= 3 or (length > 1 and s[i] == "0") or val > 255:
            print("  => nhóm sai"); return False
        groups += 1
        if j == n:
            print(f"  => hết chuỗi, groups = {groups}"); return groups == 4
        if s[j] != "." or groups == 4:
            print(f"  => ký tự {s[j]!r} tại {j} không hợp lệ hoặc thừa nhóm"); return False
        i = j + 1

print(trace_scan_ipv4("172.16.254.1"))
print(trace_scan_ipv4("1.2.3.4.5"))

  nhóm #1: s[0:3] = '172', dài 3, giá trị 172
  nhóm #2: s[4:6] = '16', dài 2, giá trị 16
  nhóm #3: s[7:10] = '254', dài 3, giá trị 254
  nhóm #4: s[11:12] = '1', dài 1, giá trị 1
  => hết chuỗi, groups = 4
True
  nhóm #1: s[0:1] = '1', dài 1, giá trị 1
  nhóm #2: s[2:3] = '2', dài 1, giá trị 2
  nhóm #3: s[4:5] = '3', dài 1, giá trị 3
  nhóm #4: s[6:7] = '4', dài 1, giá trị 4
  => ký tự '.' tại 7 không hợp lệ hoặc thừa nhóm
False


## 9. Code nộp LeetCode

Dùng Cách 2 (`split` + kiểm tra từng ký tự): dễ đọc, dễ chứng minh, `O(n)`.

In [20]:
class Solution:
    def validIPAddress(self, queryIP: str) -> str:
        digits = "0123456789"
        hexs = "0123456789abcdefABCDEF"

        def is_ipv4(s: str) -> bool:
            parts = s.split(".")
            if len(parts) != 4:
                return False
            for t in parts:
                if not 1 <= len(t) <= 3:                  # nhóm rỗng hoặc quá dài
                    return False
                if any(c not in digits for c in t):       # chỉ chấp nhận chữ số 0-9
                    return False
                if len(t) > 1 and t[0] == "0":            # không có số 0 đứng đầu
                    return False
                if int(t) > 255:
                    return False
            return True

        def is_ipv6(s: str) -> bool:
            parts = s.split(":")
            if len(parts) != 8:
                return False
            for t in parts:
                if not 1 <= len(t) <= 4:
                    return False
                if any(c not in hexs for c in t):
                    return False
            return True

        if is_ipv4(queryIP):
            return "IPv4"
        if is_ipv6(queryIP):
            return "IPv6"
        return "Neither"


for q, expected in examples:
    print(q, Solution().validIPAddress(q), expected)
    assert Solution().validIPAddress(q) == expected

172.16.254.1 IPv4 IPv4
2001:0db8:85a3:0:0:8A2E:0370:7334 IPv6 IPv6
256.256.256.256 Neither Neither


## 10. Vì sao thuật toán đúng

**Hai bộ kiểm tra loại trừ nhau.** Một chuỗi hợp lệ IPv4 chứa đúng 3 dấu `.` và không có `:` (mọi nhóm chỉ chữ số); một chuỗi hợp lệ IPv6 chứa đúng 7 dấu `:` và không có `.` (mọi nhóm chỉ hex). Hai tập đó **rời nhau**, nên thứ tự gọi `is_ipv4` rồi `is_ipv6` không ảnh hưởng.

**`is_ipv4` đúng.** Theo đề, địa chỉ IPv4 hợp lệ ⇔ có đúng bốn nhóm cách nhau bởi `.`, mỗi nhóm là biểu diễn thập phân **không số 0 đầu** của số trong `[0, 255]`. Ta kiểm tra đúng từng điều kiện: `split(".")` cho đúng nhóm theo dấu chấm (nhóm rỗng xuất hiện khi có dấu chấm thừa); độ dài 1–3 vừa loại nhóm rỗng vừa chặn số lớn (≤ 999, rồi kiểm giá trị); toàn chữ số bảo đảm `int` không lỗi và không dùng được chuỗi lạ; "không 0 đầu" nghĩa là nếu dài hơn 1 thì ký tự đầu khác `0`; cuối cùng giá trị ≤ 255. Chiều ngược lại hiển nhiên.

**`is_ipv6` đúng.** IPv6 hợp lệ ⇔ đúng 8 nhóm, mỗi nhóm 1–4 ký tự hex (0 đầu được phép). Ta kiểm đúng những điều kiện đó.

**Không còn trường hợp nào bị bỏ sót:** bất kỳ chuỗi nào không thỏa hai bộ trên không là địa chỉ hợp lệ của loại nào ⇒ `Neither`. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(queryIP)`.

| Cách | Time | Space |
|---|---|---|
| 1. `int()` (có lỗi) | `O(n)` | `O(n)` |
| 2. `split` + kiểm tra ký tự | `O(n)` | `O(n)` |
| 3. Regex `fullmatch` | `O(n)` | `O(1)` phụ |
| 4. Máy trạng thái quét một lượt | `O(n)` | `O(1)` |
| 5. `ipaddress` + lọc | `O(n)` | `O(n)` |

Mỗi nhóm IPv4 chỉ dài tối đa 3 ký tự và `int` chạy trên chuỗi tối đa 3 ký tự nên là `O(1)`; chuỗi dài hơn bị loại nhanh. Không có cách nào "tốt hơn `O(n)`" vì phải đọc hết chuỗi.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [21]:
sol = Solution()
v = sol.validIPAddress

for q, expected in examples:
    assert v(q) == expected

casos = {
    # IPv4 hợp lệ
    "0.0.0.0": "IPv4", "255.255.255.255": "IPv4", "192.168.1.0": "IPv4", "1.1.1.1": "IPv4",
    # IPv4 sai
    "192.168.01.1": "Neither", "192.168.1.00": "Neither", "192.168@1.1": "Neither",
    "1.1.1": "Neither", "1.1.1.1.1": "Neither", "1.1.1.1.": "Neither", ".1.1.1.1": "Neither",
    "1..1.1": "Neither", "256.1.1.1": "Neither", "1.1.1.256": "Neither", "1.1.1.-1": "Neither",
    "1.1.1.1a": "Neither", "0000.1.1.1": "Neither", "00.0.0.0": "Neither", "1e1.1.1.1": "Neither",
    # IPv6 hợp lệ
    "2001:0db8:85a3:0000:0000:8a2e:0370:7334": "IPv6",
    "2001:db8:85a3:0:0:8A2E:0370:7334": "IPv6",
    "0:0:0:0:0:0:0:0": "IPv6", "FFFF:ffff:FfFf:0000:1:2:3:4": "IPv6",
    # IPv6 sai
    "2001:0db8:85a3::8A2E:037j:7334": "Neither",
    "02001:0db8:85a3:0000:0000:8a2e:0370:7334": "Neither",
    "1:2:3:4:5:6:7": "Neither", "1:2:3:4:5:6:7:8:9": "Neither", "1:2:3:4:5:6:7:8:": "Neither",
    ":1:2:3:4:5:6:7": "Neither", "1:2:3:4:5:6:7:g": "Neither", "0x1f:0db8:85a3:0:0:8A2E:0370:7334": "Neither",
    "::::::::": "Neither",
    # khác
    "": "Neither", "1.1.1.1:2:3:4:5:6:7": "Neither", "abc": "Neither", ".": "Neither", ":": "Neither",
    "1.2.3.4:5": "Neither",
}
for q, expected in casos.items():
    assert v(q) == expected, (q, v(q), expected)
print("Ví dụ và", len(casos), "edge case OK")

Ví dụ và 38 edge case OK


### 12.2. Phản ví dụ cho lời giải sai

Các chuỗi sau làm vỡ các lời giải làm ẩu:

- `"0x1f:..."` làm vỡ Cách 1 (`int(t, 16)`);
- `"01.1.1.1"` làm vỡ lời giải quên số 0 đầu;
- `"1.1.1.1."` làm vỡ lời giải đếm dấu `.` mà không `split` (đếm đúng 3 dấu? không: có 4 dấu, nhưng `"1..1.1"` có đúng 3 dấu mà nhóm rỗng).

In [22]:
def sai_quen_0_dau(q):
    parts = q.split(".")
    return len(parts) == 4 and all(p.isdigit() and int(p) <= 255 for p in parts)

def sai_chi_dem_dau(q):
    return q.count(".") == 3

for q in ["01.1.1.1", "1..1.1", "1.1.1.a"]:
    print(f"{q!r:12} quên_0_đầu={sai_quen_0_dau(q)!s:5} chỉ_đếm_dấu={sai_chi_dem_dau(q)!s:5} đúng={v(q)}")
assert sai_quen_0_dau("01.1.1.1") and v("01.1.1.1") == "Neither"
assert sai_chi_dem_dau("1..1.1") and v("1..1.1") == "Neither"
assert naive("0x1f:0db8:85a3:0:0:8A2E:0370:7334") == "IPv6" and v("0x1f:0db8:85a3:0:0:8A2E:0370:7334") == "Neither"

'01.1.1.1'   quên_0_đầu=True  chỉ_đếm_dấu=True  đúng=Neither
'1..1.1'     quên_0_đầu=False chỉ_đếm_dấu=True  đúng=Neither
'1.1.1.a'    quên_0_đầu=False chỉ_đếm_dấu=True  đúng=Neither


### 12.3. Đối chiếu các cách: sinh địa chỉ hợp lệ có chủ đích

Sinh ngẫu nhiên (seed cố định) các địa chỉ **hợp lệ theo thiết kế** (IPv4: số 0–255 viết chuẩn; IPv6: 8 nhóm 16-bit viết hex ngẫu nhiên hoa/thường, đệm 0 ngẫu nhiên đến tối đa 4 ký tự) rồi kiểm tra mọi cách đều nhận đúng loại.

In [23]:
random.seed(468)

def sinh_ipv4():
    return ".".join(str(random.randint(0, 255)) for _ in range(4))

def sinh_ipv6():
    nhom = []
    for _ in range(8):
        x = random.randint(0, 0xFFFF)
        h = format(x, "x")
        h = h.rjust(random.randint(len(h), 4), "0")       # đệm 0 đầu tùy ý (được phép ở IPv6)
        nhom.append(h.upper() if random.random() < 0.5 else h)
    return ":".join(nhom)

cac_cach = {
    "split": split_check, "regex": regex_check, "scan": scan_check, "lib": lib_check,
    "Solution": Solution().validIPAddress, "Regex": SolutionRegex().validIPAddress,
    "Scan": SolutionScan().validIPAddress, "Lib": SolutionLib().validIPAddress,
    "Split": SolutionSplit().validIPAddress,
}
for _ in range(2000):
    a, b = sinh_ipv4(), sinh_ipv6()
    for ten, fn in cac_cach.items():
        assert fn(a) == "IPv4", (ten, a)
        assert fn(b) == "IPv6", (ten, b)
print("2000 cặp địa chỉ hợp lệ: tất cả các cách đều nhận đúng")

2000 cặp địa chỉ hợp lệ: tất cả các cách đều nhận đúng


### 12.4. Đối chiếu các cách với oracle: đột biến ngẫu nhiên

Lấy địa chỉ hợp lệ, **đột biến** ngẫu nhiên (đổi/xóa/chèn ký tự trong bảng chữ cho phép của đề) để có nhiều chuỗi "gần hợp lệ" khó chịu. Oracle độc lập là `lib_check` (dựng trên thư viện `ipaddress`); các cách còn lại phải cho kết quả giống hệt (riêng Cách 1 `naive` đối chiếu riêng ở dưới).

In [24]:
random.seed(4680)
ALPHA = "0123456789abcdefghxXABCDEFGH.:"

def dot_bien(q):
    q = list(q)
    for _ in range(random.randint(1, 3)):
        op = random.choice("ird")
        if op == "i" or not q:
            q.insert(random.randint(0, len(q)), random.choice(ALPHA))
        elif op == "r":
            q[random.randrange(len(q))] = random.choice(ALPHA)
        else:
            del q[random.randrange(len(q))]
    return "".join(q)

so_loai = {"IPv4": 0, "IPv6": 0, "Neither": 0}
for _ in range(6000):
    goc = sinh_ipv4() if random.random() < 0.5 else sinh_ipv6()
    q = dot_bien(goc) if random.random() < 0.9 else "".join(random.choice(ALPHA) for _ in range(random.randint(0, 12)))
    oracle = lib_check(q)
    so_loai[oracle] += 1
    for ten, fn in cac_cach.items():
        assert fn(q) == oracle, (ten, q, fn(q), oracle)
print("Phân bố kết quả oracle:", so_loai)
assert min(so_loai.values()) > 100        # đủ cả ba loại để test có ý nghĩa

Phân bố kết quả oracle: {'IPv4': 436, 'IPv6': 765, 'Neither': 4799}


In [25]:
# Cách 1 (naive): đối chiếu chỉ ở IPv4 (đúng), còn IPv6 sai ở các chuỗi có 'x' hoặc ký tự lạ nhưng int() chấp nhận
sai_lech = []
random.seed(99)
for _ in range(20000):
    goc = sinh_ipv6()
    q = dot_bien(goc)
    if random.random() < 0.3:                       # đột biến có chủ đích: chèn 'x' vào ký tự thứ 2 của một nhóm 4 ký tự
        nhom = goc.split(":")
        i = random.randrange(8)
        if len(nhom[i]) == 4:
            nhom[i] = nhom[i][0] + "x" + nhom[i][2:]
            q = ":".join(nhom)
    if naive(q) != lib_check(q):
        sai_lech.append(q)
print("Cách 1 sai trên", len(sai_lech), "/ 20000 chuỗi; ví dụ:", sai_lech[:2])
# chỉ có thể sai theo kiểu "naive nhận IPv6, đúng phải là Neither"
assert sai_lech and all(naive(q) == "IPv6" and lib_check(q) == "Neither" for q in sai_lech)
for q in ["0x1f:0db8:85a3:0:0:8A2E:0370:7334"]:
    assert naive(q) != lib_check(q)

Cách 1 sai trên 202 / 20000 chuỗi; ví dụ: ['7aa9:3acf:24f0:0x04:A35F:45B5:1873:9595', '5152:C47F:C204:A1B5:2e3b:0x28:97AD:6FE3']


### 12.5. Chuỗi rất dài

In [26]:
dai = "1." * 50_000 + "1"
t0 = time.perf_counter()
r1 = v(dai)
r2 = v("f:" * 50_000 + "f")
r3 = v("1" * 100_000)
print(r1, r2, r3, f"{1000 * (time.perf_counter() - t0):.1f} ms")
assert (r1, r2, r3) == ("Neither", "Neither", "Neither")

Neither Neither Neither 1.0 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `int(t, 16)` để kiểm tra hex

`int("0x1f", 16)` cho `31` không lỗi (mục 3.3), nên `"0x1f"` bị coi là nhóm hex hợp lệ.

In [27]:
t = "0x1f"
try:
    print("int ok:", int(t, 16))
except ValueError:
    print("int lỗi")
print("đúng phải là sai:", all(c in HEX for c in t))
assert int(t, 16) == 31 and not all(c in HEX for c in t)

int ok: 31
đúng phải là sai: False


**Vì sao sai:** `int` chấp nhận nhiều định dạng ngoài hex thuần (`0x`, `+`, `_`, khoảng trắng). **Cách đúng:** kiểm tra từng ký tự thuộc `0123456789abcdefABCDEF` rồi mới (nếu cần) đọc số.

### Lỗi 2 — Quên rằng IPv4 không được có số 0 đứng đầu

`"192.168.01.1"` thường bị `int("01") = 1 <= 255` cho qua.

In [28]:
def loi2_sai(q):
    p = q.split(".")
    return len(p) == 4 and all(x.isdigit() and 0 <= int(x) <= 255 for x in p)

print("sai :", loi2_sai("192.168.01.1"), "(mong đợi False)")
print("đúng:", v("192.168.01.1"))
assert loi2_sai("192.168.01.1") is True and v("192.168.01.1") == "Neither"

sai : True (mong đợi False)
đúng: Neither


**Vì sao sai:** `int("01")` bằng `1` nên lọt. **Cách đúng:** `len(t) > 1 and t[0] == "0"` → sai. Chú ý chính nhóm `"0"` (độ dài 1) vẫn hợp lệ.

### Lỗi 3 — Nhầm: IPv6 **cho phép** số 0 đứng đầu

Ngược với IPv4: `"0db8"` hợp lệ, `"0000"` hợp lệ. Áp quy tắc "không 0 đầu" của IPv4 cho IPv6 sẽ loại nhầm.

In [29]:
def loi3_sai(q):
    p = q.split(":")
    return len(p) == 8 and all(1 <= len(t) <= 4 and (len(t) == 1 or t[0] != "0") and all(c in HEX for c in t) for t in p)

q = "2001:0db8:85a3:0000:0000:8a2e:0370:7334"
print("sai :", loi3_sai(q), "(mong đợi True)")
print("đúng:", v(q))
assert loi3_sai(q) is False and v(q) == "IPv6"

sai : False (mong đợi True)
đúng: IPv6


**Vì sao sai:** nhầm luật giữa hai loại địa chỉ. **Cách đúng:** viết hai hàm kiểm tra tách bạch và đọc kỹ đề.

### Lỗi 4 — Bỏ sót nhóm rỗng khi có dấu phân tách thừa

Chỉ kiểm tra "số dấu `.` bằng 3" hoặc dùng `filter(None, ...)` để bỏ nhóm rỗng đều làm `"1..1.1"` hoặc `"1.1.1.1."` được nhận nhầm.

In [30]:
def loi4_sai(q):
    p = [x for x in q.split(".") if x]          # SAI: vứt nhóm rỗng đi
    return len(p) == 4 and all(x.isdigit() and int(x) <= 255 and (len(x) == 1 or x[0] != "0") for x in p)

for q in ["1.1.1.1.", ".1.1.1.1", "1..1.1.1"]:
    print(f"{q!r:12} sai={loi4_sai(q)!s:5} đúng={v(q)}")
    assert loi4_sai(q) is True and v(q) == "Neither"

'1.1.1.1.'   sai=True  đúng=Neither
'.1.1.1.1'   sai=True  đúng=Neither
'1..1.1.1'   sai=True  đúng=Neither


**Vì sao sai:** nhóm rỗng cũng là một nhóm (phải làm kết quả `False`). **Cách đúng:** kiểm tra `len(t) >= 1` cho mọi nhóm, **không lọc bỏ**.

### Lỗi 5 — Dùng `isdigit()` thay cho kiểm tra tập ký tự

`"²".isdigit()` là `True` nhưng `int("²")` ném `ValueError`: chương trình bị crash thay vì trả `"Neither"`.

In [31]:
def loi5_sai(t):
    return t.isdigit() and int(t) <= 255          # có thể ném ValueError với '²'

try:
    loi5_sai("²")
except ValueError as e:
    print("Crash:", e)
print("đúng:", all(c in CHU_SO for c in "²"))
assert all(c in CHU_SO for c in "²") is False

Crash: invalid literal for int() with base 10: '²'
đúng: False


**Vì sao sai:** `isdigit` rộng hơn `0-9`. **Cách đúng:** `c in "0123456789"`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "kiểm tra một chuỗi có đúng định dạng X không" → tách theo dấu phân tách rồi kiểm từng **trường** (field) với luật riêng.
- **Mẹo:** viết hàm **trả `False` sớm** (early return) cho từng điều kiện: code dài nhưng phẳng, ít lồng nhau.
- **Thứ tự kiểm tra:** độ dài trước (loại rỗng), rồi tập ký tự, rồi luật tinh vi (0 đầu), rồi giá trị. Gọi `int` sau cùng và chỉ trên dữ liệu đã sạch.
- **Đừng tin vào hàm chuyển đổi để kiểm tra cú pháp:** `int`, `float`, `isdigit`, `isalnum` đều rộng hơn tập bạn nghĩ.
- **Regex rất hợp** cho bài kiểu này nhưng chỉ khi bạn hiểu chắc từng thành phần; trong phỏng vấn nên nói được cả hai cách.
- **Test thật nhiều chuỗi "gần hợp lệ"** (đột biến một ký tự): đó là nơi phần lớn lỗi nằm.

## 15. Biến thể

**Biến thể A — IPv4 cho phép số 0 đứng đầu** (mỗi nhóm 1–3 chữ số, giá trị ≤ 255): bỏ điều kiện "không 0 đầu".

**Biến thể B — kiểm tra CIDR** `a.b.c.d/m` với `0 <= m <= 32`: tách bằng `/` thành hai phần, phần đầu kiểm IPv4, phần sau là số nguyên không 0 đầu trong `[0, 32]`.

**Biến thể C — đếm số địa chỉ hợp lệ trong danh sách / phân loại hàng loạt:** dùng `collections.Counter` trên kết quả của `Solution().validIPAddress`.

In [32]:
from collections import Counter

def ipv4_cho_phep_0_dau(q):
    p = q.split(".")
    return len(p) == 4 and all(1 <= len(t) <= 3 and all(c in CHU_SO for c in t) and int(t) <= 255 for t in p)

def la_cidr(q):
    if q.count("/") != 1:
        return False
    ip, m = q.split("/")
    if not is_ipv4(ip):
        return False
    return 1 <= len(m) <= 2 and all(c in CHU_SO for c in m) and (len(m) == 1 or m[0] != "0") and int(m) <= 32

print(ipv4_cho_phep_0_dau("192.168.01.1"), ipv4_cho_phep_0_dau("1.1.1.1000"))
print(la_cidr("10.0.0.0/8"), la_cidr("10.0.0.0/33"), la_cidr("10.0.0.0/08"), la_cidr("10.0.0/8"))
print(Counter(v(q) for q in ["1.1.1.1", "::", "f:f:f:f:f:f:f:f", "1.1.1", "255.0.0.255"]))
assert ipv4_cho_phep_0_dau("192.168.01.1") and not ipv4_cho_phep_0_dau("1.1.1.1000")
assert la_cidr("10.0.0.0/8") and not la_cidr("10.0.0.0/33") and not la_cidr("10.0.0.0/08") and not la_cidr("10.0.0/8")

True False
True False False False
Counter({'IPv4': 2, 'Neither': 2, 'IPv6': 1})


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `int()` | Để Python "đọc số" rồi kiểm giá trị | `O(n)` | `O(n)` | **Không dùng** (sai với `0x1f`); chỉ để học bẫy |
| 2. `split` + ký tự | Tách nhóm, kiểm tập ký tự, độ dài, 0 đầu, ≤ 255 | `O(n)` | `O(n)` | **Cách nộp bài**: rõ ràng nhất |
| 3. Regex | Mô tả hình dạng bằng mẫu, `fullmatch` | `O(n)` | `O(1)` | Muốn ngắn gọn; chắc tay về regex |
| 4. Máy trạng thái | Quét một lượt, không `split` | `O(n)` | `O(1)` | Chuỗi cực dài, không muốn cấp phát danh sách |
| 5. `ipaddress` + lọc | Thư viện chuẩn sau khi lọc hình dạng | `O(n)` | `O(n)` | Làm oracle đối chiếu; ứng dụng thật |

In [33]:
random.seed(12)
bo = [sinh_ipv4() if i % 2 == 0 else sinh_ipv6() for i in range(20000)]
bo += [dot_bien(q) for q in bo[:20000]]
cac = [("split", split_check), ("regex", regex_check), ("scan", scan_check), ("lib", lib_check), ("naive", naive)]
for ten, fn in cac:
    t0 = time.perf_counter()
    ket_qua = [fn(q) for q in bo]
    print(f"{ten:<6} {1000 * (time.perf_counter() - t0):7.1f} ms  IPv4={ket_qua.count('IPv4')} IPv6={ket_qua.count('IPv6')} Neither={ket_qua.count('Neither')}")

split     58.0 ms  IPv4=11704 IPv6=12906 Neither=15390


regex     16.1 ms  IPv4=11704 IPv6=12906 Neither=15390


scan      64.9 ms  IPv4=11704 IPv6=12906 Neither=15390
lib      113.1 ms  IPv4=11704 IPv6=12906 Neither=15390


naive     28.3 ms  IPv4=11704 IPv6=12907 Neither=15389


## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán `validIPAddress` của: (a) `"1.0.0.1"`; (b) `"1.0.0.01"`; (c) `"a:b:c:d:e:f:0:1"`; (d) `"1:2:3:4:5:6:7:12345"`; (e) `"256.1.1.1"`.

**Bài 2 (tính tay).** Cho `"2001:db8:85a3:0:0:8A2E:0370:7334:"`. Khi gọi `split(":")` có bao nhiêu nhóm? Nhóm cuối là gì? Kết quả của hàm?

**Bài 3 (điền TODO).** Hoàn thành `la_nhom_ipv4(t)` trả `True` nếu chuỗi `t` là một nhóm IPv4 hợp lệ.

In [34]:
def la_nhom_ipv4(t):
    # TODO 1: độ dài phải trong [1, 3]
    # TODO 2: mọi ký tự phải thuộc "0123456789"
    # TODO 3: nếu dài hơn 1 thì ký tự đầu không được là "0"
    # TODO 4: int(t) <= 255
    return None

# KHÔNG gọi la_nhom_ipv4 ở đây để "Run All" không lỗi (chưa điền xong).

**Bài 4 (tự viết lại).** Viết `kiem_tra_regex(q)` bằng **regex** mà không nhìn lại mục 7 (nhớ dùng `fullmatch`). Kiểm tra trên ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Cho phép IPv6 **viết tắt một lần `::`** (đại diện cho một hoặc nhiều nhóm toàn 0; tổng số nhóm tường minh `<= 7` khi có `::`). Viết `ipv6_co_viet_tat(q)` trả `True/False`. Gợi ý: nếu `q.count("::") == 1`, tách thành `trái` và `phải` bằng `"::"`, tách mỗi phần bằng `":"` (phần rỗng cho danh sách rỗng), kiểm tra mọi nhóm hợp lệ và `len(trái) + len(phải) <= 7`. Kiểm tra: `"::1"`, `"1::"`, `"::"`, `"1::2"` hợp lệ; `"1::2::3"`, `":::"`, `"1:2:3:4:5:6:7::8"` không hợp lệ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: (a) hợp lệ; (b) `"01"` có 0 đầu; (c) 8 nhóm hex hợp lệ; (d) nhóm `12345` dài 5; (e) `256 > 255`. Bài 3: viết 4 `if ... return False` rồi `return True`.
- **Mức 2 (vừa):** Bài 2: dấu `:` cuối sinh thêm nhóm rỗng ở cuối nên có 9 nhóm. Bài 4: IPv4 chia 4 nhóm bằng `(...\.){3}...`; thử trước từng nhóm 0–255 trong mẫu `25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9]`.
- **Mức 3 (sâu):** Bài 5: ca `"::"` cho `trái = [""]`? Hãy dùng `"::".join` cẩn thận: `q.split("::")` ra `["", ""]`; đổi phần rỗng thành `[]` bằng `p.split(":") if p else []`. Các ca có `:::` cần loại trước vì `"::"` sẽ đếm chồng.

## 19. Đáp án tham khảo cho bài tập

In [35]:
# Bài 1
assert v("1.0.0.1") == "IPv4"
assert v("1.0.0.01") == "Neither"
assert v("a:b:c:d:e:f:0:1") == "IPv6"
assert v("1:2:3:4:5:6:7:12345") == "Neither"
assert v("256.1.1.1") == "Neither"
print("Bài 1 OK")

# Bài 2
q = "2001:db8:85a3:0:0:8A2E:0370:7334:"
nhom = q.split(":")
print(len(nhom), repr(nhom[-1]), v(q))
assert len(nhom) == 9 and nhom[-1] == "" and v(q) == "Neither"
print("Bài 2 OK")

# Bài 3
def la_nhom_ipv4_ans(t):
    if not 1 <= len(t) <= 3:
        return False
    if any(c not in "0123456789" for c in t):
        return False
    if len(t) > 1 and t[0] == "0":
        return False
    return int(t) <= 255

assert [la_nhom_ipv4_ans(t) for t in ["0", "255", "256", "01", "", "1a", "1234"]] == [True, True, False, False, False, False, False]
print("Bài 3 OK")

# Bài 4
RE_Bai4_4 = re.compile(r"(25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])(\.(25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])){3}")
RE_Bai4_6 = re.compile(r"([0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}")

def kiem_tra_regex(q):
    if RE_Bai4_4.fullmatch(q):
        return "IPv4"
    return "IPv6" if RE_Bai4_6.fullmatch(q) else "Neither"

for q, expected in examples:
    assert kiem_tra_regex(q) == expected
random.seed(3)
for _ in range(500):
    q = dot_bien(sinh_ipv4() if random.random() < 0.5 else sinh_ipv6())
    assert kiem_tra_regex(q) == v(q)
print("Bài 4 OK")

# Bài 5
def ipv6_co_viet_tat(q):
    if any(c not in HEX + ":" for c in q) or ":::" in q:
        return False
    if q.count("::") != 1:
        return is_ipv6(q)
    trai, phai = q.split("::")
    a = trai.split(":") if trai else []
    b = phai.split(":") if phai else []
    nhom = a + b
    return len(nhom) <= 7 and all(1 <= len(t) <= 4 and all(c in HEX for c in t) for t in nhom)

for ok in ["::1", "1::", "::", "1::2", "1:2:3:4:5:6:7:8", "1:2:3:4:5:6::7"]:
    assert ipv6_co_viet_tat(ok), ok
for bad in ["1::2::3", ":::", "1:2:3:4:5:6:7::8", "1:2", "12345::1", "g::1"]:
    assert not ipv6_co_viet_tat(bad), bad
print("Bài 5 OK")

Bài 1 OK
9 '' Neither
Bài 2 OK
Bài 3 OK
Bài 4 OK
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (kiểm tra định dạng bằng `split` + luật theo trường):**

```python
def validIPAddress(self, queryIP):
    def is_ipv4(s):
        p = s.split(".")
        if len(p) != 4: return False
        for t in p:
            if not 1 <= len(t) <= 3 or any(c not in "0123456789" for c in t): return False
            if len(t) > 1 and t[0] == "0": return False
            if int(t) > 255: return False
        return True

    def is_ipv6(s):
        p = s.split(":")
        if len(p) != 8: return False
        return all(1 <= len(t) <= 4 and all(c in "0123456789abcdefABCDEF" for c in t) for t in p)

    return "IPv4" if is_ipv4(queryIP) else "IPv6" if is_ipv6(queryIP) else "Neither"
```

**Checklist:**

- [ ] IPv4: đúng 4 nhóm, mỗi nhóm 1–3 chữ số **thập phân** (kiểm bằng tập ký tự, không dùng `isdigit`).
- [ ] IPv4: không số 0 đứng đầu (trừ chính `"0"`), giá trị ≤ 255.
- [ ] IPv6: đúng 8 nhóm, mỗi nhóm 1–4 ký tự hex (hoa/thường), **cho phép** 0 đầu.
- [ ] Không lọc bỏ nhóm rỗng; nhóm rỗng phải làm kết quả `Neither`.
- [ ] Không dùng `int(t, 16)` để kiểm tra hex (bẫy `0x1f`).
- [ ] Thử: `"0.0.0.0"`, `"1.1.1.1."`, `"01.1.1.1"`, `"0x1f:..."`, `"::"`, chuỗi rỗng.
- [ ] Đúng signature `class Solution` / `validIPAddress(self, queryIP: str) -> str`.